In [1]:
import sqlite3
connect = sqlite3.connect("../db/pl_data.db")
cursor = connect.cursor()

In [ ]:
# confirming total games
cursor.execute("SELECT season_year, COUNT(*) FROM matches GROUP BY season_year")
cursor.fetchall()

In [ ]:
# get teams for CSV matching
cursor.execute("SELECT team_id, name FROM teams")
cursor.fetchall()

In [ ]:
cursor.execute("SELECT season_year, COUNT(*) FROM matches GROUP BY season_year")
cursor.fetchall()

In [2]:
import pandas as pd
import sqlite3

connect = sqlite3.connect("../db/pl_data.db")
matches = pd.read_sql_query("SELECT * FROM matches", connect)

matches.head()

,match_id,season_year,matchday,utc_date,status,home_team_id,away_team_id,home_goals_fulltime,away_goals_fulltime,home_goals_halftime,...,away_corners,home_yellow,away_yellow,home_red,away_red,referee,b365_home_odds,b365_draw_odds,b365_away_odds,winner
0,20000,2018,None,10/08/2018,FINISHED,66,9007,2,1,1,...,5,2,1,0,0,A Marriner,1.57,3.9,7.50,HOME_TEAM
1,20001,2018,None,11/08/2018,FINISHED,1044,9003,2,0,1,...,4,1,1,0,0,K Friend,1.90,3.6,4.50,HOME_TEAM
2,20002,2018,None,11/08/2018,FINISHED,63,354,0,2,0,...,5,1,2,0,0,M Dean,2.50,3.4,3.00,AWAY_TEAM
3,20003,2018,None,11/08/2018,FINISHED,9004,61,0,3,0,...,5,2,1,0,0,C Kavanagh,6.50,4.0,1.61,AWAY_TEAM
4,20004,2018,None,11/08/2018,FINISHED,67,73,1,2,1,...,5,2,2,0,0,M Atkinson,3.90,3.5,2.04,AWAY_TEAM


In [3]:
# home DataFrame
home = matches[["season_year", "home_team_id", "home_goals_fulltime", "away_goals_fulltime", "winner"]].copy()
home.columns = ["season_year", "team_id", "goals_for", "goals_against", "winner",]

# away DataFrame
away = matches[["season_year", "away_team_id", "away_goals_fulltime", "home_goals_fulltime", "winner"]].copy()
away.columns = ["season_year", "team_id", "goals_for", "goals_against", "winner",]

home_result_map = {
    "HOME_TEAM": "WIN",
    "AWAY_TEAM": "LOSS",
    "DRAW": "DRAW"
}

away_result_map = {
    "HOME_TEAM": "LOSS",
    "AWAY_TEAM": "WIN",
    "DRAW": "DRAW"
}

home["result"] = home["winner"].map(home_result_map)
away["result"] = away["winner"].map(away_result_map)

team_matches = pd.concat([home, away], ignore_index=True)

seasons = team_matches.groupby(["team_id", "season_year"]).agg(
    matches_played=("winner", "count"),
    total_goals_for=("goals_for", "sum"),
    total_goals_against=("goals_against", "sum"),
    wins=("result", lambda x: (x == "WIN").sum()),
    draws=("result", lambda x: (x == "DRAW").sum()),
    losses=("result", lambda x: (x == "LOSS").sum()),
)

seasons["points"] = seasons["wins"] * 3 + seasons["draws"]
seasons["goal_difference"] = seasons["total_goals_for"] - seasons["total_goals_against"] 
# seasons["position"] = seasons.groupby("season_year")["points"].rank(ascending=False)

seasons = seasons.sort_values(["season_year", "points", "goal_difference"], ascending=[True, False, False])
seasons["position"] = seasons.groupby("season_year").cumcount() + 1



seasons = seasons[["matches_played", "total_goals_for", "total_goals_against", "goal_difference", "wins", "draws", "losses", "points", "position"]]

In [4]:
seasons.head(21)

,,matches_played,total_goals_for,total_goals_against,goal_difference,wins,draws,losses,points,position
team_id,season_year,,,,,,,,,
65,2018,38,95,23,72,32,2,4,98,1
64,2018,38,89,22,67,30,7,1,97,2
61,2018,38,63,39,24,21,9,8,72,3
73,2018,38,67,39,28,23,2,13,71,4
57,2018,38,73,51,22,21,7,10,70,5
66,2018,38,65,54,11,19,9,10,66,6
76,2018,38,47,46,1,16,9,13,57,7
62,2018,38,54,46,8,15,9,14,54,8
9007,2018,38,51,48,3,15,7,16,52,9


In [6]:
seasons_flat = seasons.reset_index()
seasons_flat.head()

,team_id,season_year,matches_played,total_goals_for,total_goals_against,goal_difference,wins,draws,losses,points,position
0,65,2018,38,95,23,72,32,2,4,98,1
1,64,2018,38,89,22,67,30,7,1,97,2
2,61,2018,38,63,39,24,21,9,8,72,3
3,73,2018,38,67,39,28,23,2,13,71,4
4,57,2018,38,73,51,22,21,7,10,70,5


In [97]:
seasons[seasons["position"] % 1 != 0].shape

(0, 9)